## Preprocessing

Input: `data/processed/train.parquet`, `data/processed/test.parquet` (saved từ `demo.ipynb`)

Output: `data/processed/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

Các bước:
1. Fill NaN trong dynamic features (VLE + assessment)
2. Impute `imd_band = '?'` theo phân phối từ train
3. Encode categorical features
4. Lưu feature matrix

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import DB_PATH, SNAPSHOTS, STATIC_FEATURES, RANDOM_SEED

import numpy as np
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_parquet('../data/processed/train.parquet')
test  = pd.read_parquet('../data/processed/test.parquet')

# Cần assessment để tính lại num_due trong fill_in_assessment
conn = sqlite3.connect(DB_PATH)
assessment = pd.read_sql("SELECT * FROM assessments", conn)
conn.close()

print('train:', train.shape)
print('test: ', test.shape)

train: (142384, 22)
test:  (35910, 22)


---

### 1. Fill NaN — Dynamic features

In [3]:
from src.features.build_features import fill_in_weekly_clicks, fill_in_assessment

train = fill_in_weekly_clicks(train)
train = fill_in_assessment(train, assessment)

test = fill_in_weekly_clicks(test)
test = fill_in_assessment(test, assessment)

In [4]:
# avg_score và avg_days_early vẫn NaN khi sinh viên chưa nộp bài nào
# → fill 0: no_submission_despite_due đã capture trường hợp này riêng
for df in [train, test]:
    df[['avg_score', 'avg_days_early']] = df[['avg_score', 'avg_days_early']].fillna(0)

dynamic_features = [
    'total_clicks', 'active_weeks', 'avg_weekly_clicks',
    'num_due', 'num_submitted', 'avg_score', 'submission_rate',
    'num_failed', 'avg_days_early', 'no_submission_despite_due',
]

null_counts = pd.DataFrame({
    'train': train[dynamic_features].isna().sum(),
    'test':  test[dynamic_features].isna().sum(),
})
print(null_counts[null_counts.any(axis=1)])

                 train  test
submission_rate   2017   535


---

### 2. Impute `imd_band = '?'`

Fit phân phối trên train (theo từng region), áp dụng cho cả train và test.

In [5]:
def fit_imd_distributions(train_df):
    distributions = {}
    non_missing = train_df[train_df['imd_band'] != '?']
    for region, group in non_missing.groupby('region'):
        distributions[region] = group['imd_band'].value_counts(normalize=True)
    distributions['__overall__'] = non_missing['imd_band'].value_counts(normalize=True)
    return distributions


def apply_imd_imputation(df, distributions, rng):
    df = df.copy()
    for region, dist in distributions.items():
        if region == '__overall__':
            continue
        mask = (df['region'] == region) & (df['imd_band'] == '?')
        n = mask.sum()
        if n > 0:
            df.loc[mask, 'imd_band'] = rng.choice(dist.index, size=n, p=dist.values)
    still_missing = df['imd_band'] == '?'
    if still_missing.any():
        overall = distributions['__overall__']
        df.loc[still_missing, 'imd_band'] = rng.choice(
            overall.index, size=still_missing.sum(), p=overall.values
        )
    return df


imd_distributions = fit_imd_distributions(train)
rng = np.random.default_rng(RANDOM_SEED)

train = apply_imd_imputation(train, imd_distributions, rng)
test  = apply_imd_imputation(test,  imd_distributions, rng)

print('imd_band ? còn lại — train:', (train['imd_band'] == '?').sum(),
      '| test:', (test['imd_band'] == '?').sum())

imd_band ? còn lại — train: 0 | test: 0


---

### 3. Encode categorical features

| Feature | Kiểu | Lý do |
|---|---|---|
| `disability` | Binary | Y→1, N→0 |
| `highest_education` | Ordinal | Có thứ tự theo bậc học |
| `imd_band` | Ordinal | Có thứ tự theo mức độ nghèo |

> `gender`, `region`, `age_band`, `total_clicks`, `prediction_point` bị loại khỏi feature matrix (xem `DROP_COLS`). Encode vẫn chạy để giữ train/test sạch, nhưng các cột đó không đưa vào X.

In [6]:
# Binary
DISAB_MAP = {'Y': 1, 'N': 0}

# Ordinal
EDU_ORDER = {
    'No Formal quals':             0,
    'Lower Than A Level':          1,
    'A Level or Equivalent':       2,
    'HE Qualification':            3,
    'Post Graduate Qualification': 4,
}

# '10-20' (không có %) là cách ghi gốc trong OULAD
IMD_ORDER = {
    '0-10%': 0, '10-20': 1, '20-30%': 2, '30-40%': 3, '40-50%': 4,
    '50-60%': 5, '60-70%': 6, '70-80%': 7, '80-90%': 8, '90-100%': 9,
}

for df in [train, test]:
    df['disability']        = df['disability'].map(DISAB_MAP)
    df['highest_education'] = df['highest_education'].map(EDU_ORDER)
    df['imd_band']          = df['imd_band'].map(IMD_ORDER)

# Label encode target — fit trên train
labels = sorted(train['final_result'].unique())
LABEL_MAP = {l: i for i, l in enumerate(labels)}
print('Label map:', LABEL_MAP)

Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}


---

### 4. Build feature matrices

In [7]:
DROP_COLS = {'gender', 'region', 'age_band', 'total_clicks'}
# prediction_point không phải feature — không đưa vào X, lưu riêng để modeling tách theo mốc
FEATURE_COLS = [c for c in STATIC_FEATURES + dynamic_features if c not in DROP_COLS]

X_train = train[FEATURE_COLS].copy()
X_test  = test[FEATURE_COLS].copy()

y_train = train['final_result'].map(LABEL_MAP)
y_test  = test['final_result'].map(LABEL_MAP)

print('X_train:', X_train.shape, '| NaN:', X_train.isna().sum().sum())
print('X_test: ', X_test.shape,  '| NaN:', X_test.isna().sum().sum())
print('y_train:\n', y_train.value_counts().sort_index())
print('y_test:\n',  y_test.value_counts().sort_index())
print('\nFeatures:', FEATURE_COLS)

X_train: (142384, 14) | NaN: 2017
X_test:  (35910, 14) | NaN: 535
y_train:
 final_result
0    40333
1    88536
2    13515
Name: count, dtype: int64
y_test:
 final_result
0    10192
1    22432
2     3286
Name: count, dtype: int64

Features: ['imd_band', 'highest_education', 'num_of_prev_attempts', 'studied_credits', 'disability', 'active_weeks', 'avg_weekly_clicks', 'num_due', 'num_submitted', 'avg_score', 'submission_rate', 'num_failed', 'avg_days_early', 'no_submission_despite_due']


In [8]:
out_dir = Path('../data/processed')
out_dir.mkdir(exist_ok=True)

X_train.to_parquet(out_dir / 'X_train.parquet', index=False)
X_test.to_parquet(out_dir  / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(out_dir / 'y_train.parquet', index=False)
y_test.to_frame().to_parquet(out_dir  / 'y_test.parquet',  index=False)

# Lưu prediction_point riêng để modeling dùng tách dữ liệu theo từng mốc
train[['prediction_point']].to_parquet(out_dir / 'train_meta.parquet', index=False)
test[['prediction_point']].to_parquet(out_dir  / 'test_meta.parquet',  index=False)

print('Saved to data/processed/')
print('Label map:', LABEL_MAP)
print('Feature cols:', FEATURE_COLS)

Saved to data/processed/
Label map: {'Fail': 0, 'Pass': 1, 'Withdrawn': 2}
Feature cols: ['imd_band', 'highest_education', 'num_of_prev_attempts', 'studied_credits', 'disability', 'active_weeks', 'avg_weekly_clicks', 'num_due', 'num_submitted', 'avg_score', 'submission_rate', 'num_failed', 'avg_days_early', 'no_submission_despite_due']
